# Number of Target Estimation with AIC and MDL

A significant number of DoA estimation methods are required to know 

In [1]:
# Importing modules
import numpy as np; 
import matplotlib.pyplot as plt; 

In [5]:
# Defining methods
hermitian = lambda array: np.conj(array).T; 


def g_mean(arr): # Geometric mean function
  g = 1; 
  for i in range(len(arr)):
    g = g*arr[i]; 
  return g**(1/len(arr)); 


def a_mean(arr): # Arithmetic mean function
  a = 0; 
  for i in range(len(arr)):
    a = a+arr[i]; 
  return a/len(arr); 


def steering_vector(sensor_pos: np.ndarray, ang_elev: float, ang_azim: float, wavelen: float) -> np.ndarray:
  omega = np.array([[np.sin(ang_elev)*np.cos(ang_azim)],
                    [np.sin(ang_elev)*np.sin(ang_azim)],
                    [np.cos(ang_elev)]]); 
  return np.exp(1j*2*np.pi/wavelen*sensor_pos.T@omega); 


def generate_pos_1d_ula(
    N: int,
    d: float,
    axis=(1.,0.,0.),
    x_init=(0.,0.,0.)
    ) -> np.ndarray:
  if len(axis) != 3:
    raise TypeError(f"""The axis argument represents a 3D Cartesian vector.
      The length of the input ({len(axis)}) does not match with expected
      size 3."""); 
  if sum(axis) != 1:
    axis_new = (x/sum(axis) for x in axis); 
    axis = axis_new; 

  if len(x_init) != 3:
    raise TypeError(f"""The x_init argument represents a 3D cartesian vector.
      The length of the input ({len(x_init)}) does not match with expected
      size 3."""); 

  sensor_pos = np.tile(x_init, N).reshape(N,3).T \
    + (np.arange(0,N,1)*d) * np.tile(np.array(axis), N).reshape(N,3).T; 
  return sensor_pos; 


def generate_random_targets(ang_min: float, ang_max: float, ang_dist: float, K: int) -> np.ndarray:
  is_valid = False; 
  while not is_valid:
    angs = np.random.uniform(ang_min, ang_max, K); 
    angs_sort = np.sort(angs); 
    for i in range(K-1):  # break for if is not valid
      is_valid = np.abs(angs_sort[i+1] - angs_sort[i]) >= ang_dist; 
  return angs; 

## Benchmark Methods: MVDR, MUSIC, ESPRIT

In this example, we will compare the behavior of MVDR with that of MUSIC and ESPRIT in the target number mismatch scenario. We already established that MUSIC and ESPRIT depend on the exact and true knowledge of $K$. We will now investigate the scenarios where this is not the case.

In [6]:
# Benchmark methods
def doa_est_capon(theta_scan: np.ndarray, sensor_pos: np.ndarray, R_xx: np.ndarray, wl: float) -> np.ndarray:
  Rxx_inv = np.linalg.inv(R_xx); 
  P_capon = np.zeros(len(theta_scan), dtype=np.complex128); 

  for i, theta in enumerate(theta_scan):
    bartlett = steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl); 
    capon = (Rxx_inv @ bartlett) / (hermitian(bartlett) @ Rxx_inv @ bartlett); 
    P_capon[i] = np.squeeze(hermitian(capon) @ R_xx @ capon); 

  return P_capon; 


def doa_est_music(theta_scan: np.ndarray, sensor_pos:  np.ndarray, R_xx: np.ndarray, wl: float, K: int):
  _, e_vec = np.linalg.eigh(R_xx); 
  Un = e_vec[:, :-K]; 

  P_music = np.zeros(len(theta_scan), dtype=np.float64); 

  for i, theta in enumerate(theta_scan):
      a_theta = steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl); 
      denominator = np.abs(a_theta.conj().T @ Un @ Un.conj().T @ a_theta); 
      P_music[i] = 1 / np.squeeze(denominator); 

  return P_music; 

def doa_est_esprit(R_xx: np.ndarray, d: float, wl: float, K: int) -> np.ndarray:
  U, _, _ = np.linalg.svd(R_xx);      # Left singular matrix
  Us = U[:, :K];                      # Signal Subspace

  # Maximally overlapping subarrays 
  U1 = Us[:-1, :];                    # First M-1 rows (Subarray 1)
  U2 = Us[1:, :];                     # Last M-1 rows (Subarray 2)

  # Total Least Squares (TLS) solution of Psi
  U12 = np.hstack((U1, U2)); 

  _, _, Vh = np.linalg.svd(U12); 
  Vh = Vh.T.conj();                   # Get right singular vectors as columns

  V12 = Vh[:K, K:]; 
  V22 = Vh[K:, K:]; 

  Psi = -V12 @ np.linalg.inv(V22); 

  eig_vals = np.linalg.eigvals(Psi);   # Complex eigenvalues of the rotation matrix
  phases = np.angle(eig_vals);         # The phases (angles) of the eigenvalues

  estimated_angles_rad = np.arcsin(phases / (2 * np.pi * (d / wl))); 
  return np.sort(np.rad2deg(estimated_angles_rad)); # returns in degrees

In [7]:
# Defining parameters
c = 3*1e8;      # The speed of light in vacuum (m/s)
f_c = 5*1e9;    # Target carrier frequency (Hz)
f_s = 100*1e9;  # Receiver sampling frequency (Hz)
wl = c/f_c;       # Signal wavelenght (m)
d = wl/2;       # Antenna distance (m)

N = 16;         # Number of antennas
T = 100;        # Number of snapshots
K = np.random.randint(1,3*N/4);          # Number of targets (selected randomly between 1 and N/2)
symmetrization = False; 

S_db = [0]*K;   # Target signal power (dB)
snr_db = 0.0;   # Signal-to-Noise Ratio (dB)

ang_azim = 0;   # Azimuth angle (deg)
ang_min = -60;  # Minimum elevation angle (deg)
ang_max = 60;   # Maximum elevation angle (deg)
ang_dist = 5;   # Minimum distance between targets (deg)
ang_res = 0.1   # Angular resolution in scanning (deg)

sensor_pos = generate_pos_1d_ula(N, d);                                 # Sensor position
theta_scan = np.arange(-90,90+ang_res,ang_res);                         # Angle scan
true_angles = generate_random_targets(ang_min, ang_max, ang_dist, K);   # Target elevation angles

for k in range(K):
  print(f"Target {k} True Angle: {true_angles[k]:.3f}°"); 

Target 0 True Angle: -52.686°
Target 1 True Angle: 59.304°
Target 2 True Angle: 25.426°
Target 3 True Angle: -35.596°
Target 4 True Angle: 18.984°
Target 5 True Angle: -0.599°
Target 6 True Angle: -11.836°
Target 7 True Angle: 2.102°
Target 8 True Angle: 10.538°
Target 9 True Angle: -19.506°


In [9]:
# Data Generation
A = np.column_stack([steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl) for theta in true_angles]); # Array Steering Matrix
S_amp = np.diag([np.sqrt(10**(s_db / 10)) for s_db in S_db]); 
S = S_amp @ (np.random.randn(K, T) + 1j*np.random.randn(K, T))/np.sqrt(2); # Target Signals: Uncorrelated Gaussian (variance = 1.0)

# Noise: Spatially white complex Gaussian noise
noise_pow = 10**(-snr_db/10); 
Noise = np.sqrt(noise_pow) * (np.random.randn(N, T) + 1j*np.random.randn(N, T))/np.sqrt(2); 

# Received Signal at the Array
X = A @ S + Noise; 

In [10]:
Rxx_samp = (X @ X.conj().T)/T;        # Sample (estimated) covariance
eig_val, eig_vec = np.linalg.eigh(Rxx_samp); 

### Hermitian Symmetrization

In [11]:
Rxx_samp = (X @ X.conj().T)/T;        # Sample (estimated) covariance
if symmetrization:                    # Hermitian Symmetrization guarantees Hermitian matrix for lower snapshot scenarios.
  Rxx_samp = 0.5*(Rxx_samp + Rxx_samp.conj().T); 
eig_val, eig_vec = np.linalg.eigh(Rxx_samp); 

## Akaike Information Criterion (AIC)

In [12]:
# Akaike Information Criterion (AIC)
aic = np.zeros(N); 
for k in range(N):
  aic[k] = -2*T*(N-k+1)*np.log(g_mean(eig_val[:N-k])/a_mean(eig_val[:N-k])) + 2*k*(2*N-k); 

## Minimum Description Length (MDL)

In [13]:
# Minimum Description Length (MDL)
mdl = np.zeros(N); 
for k in range(N):
  mdl[k] = -T*(N-k+1)*np.log(g_mean(eig_val[:N-k])/a_mean(eig_val[:N-k])) + k*(2*N-k)*np.log(T)/2; 

In [14]:
K_aic = np.argmin(aic); 
K_mdl = np.argmin(mdl); 

print(f"True number of targets: {K}"); 
print(f"Akaike Information Criterion estimation: {K_aic}"); 
print(f"Minimum Description Length estimation: {K_mdl}"); 

True number of targets: 10
Akaike Information Criterion estimation: 10
Minimum Description Length estimation: 10
